# Teaching notebook — identifying a bound metal from an IR spectrum

**What you'll build.** A classifier that looks at the simulated infrared spectrum of a peptide
and says which of four divalent metals — Co²⁺, Ni²⁺, Cu²⁺, Zn²⁺ — is bound to it. Then you'll
ask the question this project actually cares about: *does that skill survive when the peptide
gets longer?*

**Why anyone cares.** Recovering transition metals from electronic waste means separating metals
that are chemically almost identical. One route is a peptide that binds one metal preferentially.
To design such a peptide you first need to read out which metal it caught — and IR is a cheap,
fast readout. Long peptides are the realistic target, but they are expensive to simulate, so we
want to train on short ones and transfer.

**The data.** A 360-molecule subset of the full study, committed at `data/teaching/`. Every
spectrum was simulated with the semi-empirical method xTB (GFN2). One spectrum per molecule
(the lowest-free-energy conformer), on a 791-point grid from 50 to 4000 cm⁻¹.

| | Co²⁺ | Ni²⁺ | Cu²⁺ | Zn²⁺ | apo |
|---|---|---|---|---|---|
| monomer | 20 | 20 | 20 | 20 | 20 |
| dimer | 20 | 20 | 20 | 20 | 20 |
| trimer | 40 | 40 | 40 | 40 | — |

**Runtime** is about a minute. **Prerequisites**: `pip install -e .` from the repo root.

> **How this differs from the full study.** The real pipeline represents each molecule by *all*
> of its conformers (~25,000 spectra) rather than just the lowest-energy one, and it caps and
> groups them carefully. That changes the numbers — the full study gets 0.81 in-domain and 0.63
> zero-shot, where you'll see roughly 0.76 and 0.52 — but not the story. Section 4 explains why
> the grouping matters and where this simplification would bite you.

## Setup

Notice what we import: the metal palette comes from `irspectra.viz.palette`, which is the single
place it is defined. Re-declaring `{"Cu+2": "#E69F00", ...}` in your own notebook is how two
figures in the same paper end up with different colors for copper.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from irspectra.data import processed, spectra
from irspectra.config import METAL_ORDER
from irspectra.viz.panels import annotate_bands
from irspectra.viz.palette import METAL_COLORS, METAL_LABELS, ALL_COLORS, ALL_ORDER

meta, X, sticks = processed.load_teaching()
WN = spectra.WN                      # 791 wavenumbers, 50..4000 cm^-1

print(f"{len(meta)} molecules | X {X.shape} | grid {WN.min():.0f}-{WN.max():.0f} cm^-1")
meta.groupby(["length", "metal"]).size().unstack(fill_value=0)

### What one row is

`meta` is the label table and `X` is the feature matrix; row *i* of one describes row *i* of the
other. That alignment is the whole contract — if you ever filter one, filter the other the same
way, or every label silently belongs to the wrong spectrum.

- `amino_acid` — the peptide sequence, e.g. `AC` is Ala-Cys
- `metal` — the label we want to predict (`none` = apo, i.e. no metal bound)
- `length` — 1 monomer, 2 dimer, 3 trimer. This is the *generalization axis*.
- `X[i]` — 791 intensities, already broadened and normalized so each spectrum sums to 1

In [ ]:
meta.head(3)

## Step 1 — Look at the data first

Before any modelling: does the signal even look different between classes? Below is the mean
Cu²⁺ spectrum at each peptide length, with the standard IR band assignments shaded.

Read it as a chemist. The **amide I** band (~1600–1700 cm⁻¹, the C=O stretch) and **amide II**
(~1500–1580 cm⁻¹) grow as the peptide gets longer, simply because a trimer has more backbone
carbonyls than a monomer. The **metal–ligand** region below 600 cm⁻¹ is where the metal itself
vibrates against the atoms it is bound to — that is where you would *hope* the metal identity
lives.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for length, style in [(1, "-"), (2, "--"), (3, ":")]:
    sel = (meta["metal"].values == "Cu+2") & (meta["length"].values == length)
    ax.plot(WN, X[sel].mean(axis=0), style, lw=1.8, color=METAL_COLORS["Cu+2"],
            label=f"length {length}  (n={sel.sum()})")
ax.set_xlim(WN.min(), WN.max())
ax.set_xlabel("wavenumber (cm$^{-1}$)")
ax.set_ylabel("mean normalized intensity")
ax.set_title("Step 1 — mean Cu$^{2+}$ spectrum by peptide length")
annotate_bands(ax, fontsize=8)
ax.legend()
plt.show()

**Now the harder question.** The lengths clearly differ — but do the *metals*? Here are all four
mean spectra at fixed length, zoomed into the metal–ligand region.

They overlap heavily. No single band separates them by eye; that is exactly why this is a
machine-learning problem rather than a lookup table.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (lo, hi, title) in zip(axes, [(50, 700, "metal-ligand region"),
                                      (1300, 1750, "amide / carboxylate region")]):
    band = (WN >= lo) & (WN <= hi)
    for m in METAL_ORDER:
        sel = (meta["metal"].values == m) & (meta["length"].values == 2)
        ax.plot(WN[band], X[sel][:, band].mean(axis=0), lw=1.8,
                color=METAL_COLORS[m], label=METAL_LABELS[m])
    ax.set_xlabel("wavenumber (cm$^{-1}$)")
    ax.set_title(f"dimers — {title}")
axes[0].set_ylabel("mean normalized intensity")
axes[0].legend()
plt.show()

## Step 2 — Featurization: from stick spectra to something a model can use

xTB does not return a smooth curve. It returns a **stick spectrum**: a list of discrete
vibrational modes, each with a frequency and an intensity. A real spectrometer never sees that —
thermal motion, collisions and finite resolution smear every line into a peak.

So we convolve each stick with a Gaussian of a chosen **FWHM** (full width at half maximum), then
normalize the result to unit total intensity so that a molecule with a stronger overall dipole
doesn't dominate purely on scale.

FWHM is the one free knob here, and it is a real modelling choice:
- **too narrow** — you keep numerical noise from the frequency calculation, and neighbouring
  molecules never overlap in feature space
- **too wide** — genuinely distinct bands merge and you erase the information you need

The project uses **15 cm⁻¹**. Try changing it below.

In [ ]:
i = int(np.where((meta["metal"].values == "Ni+2") & (meta["length"].values == 2))[0][0])
row = meta.iloc[i]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.vlines(WN, 0, sticks[i] / sticks[i].max(), color="0.6", lw=0.8, label="raw sticks (scaled)")
for fwhm, color in [(5.0, "#1BAF7A"), (15.0, "#C1651B"), (60.0, "#2A78D6")]:
    b = spectra.broaden_normalize(sticks[i][None, :], fwhm=fwhm)[0]
    ax.plot(WN, b / b.max(), lw=1.8, color=color, label=f"FWHM {fwhm:g} cm$^{{-1}}$")
ax.set_xlim(1200, 1800)                      # zoom where the amide bands live
ax.set_xlabel("wavenumber (cm$^{-1}$)")
ax.set_ylabel("intensity (scaled to peak)")
ax.set_title(f"Step 2 — broadening {row['amino_acid']} · {row['metal']}  (15 cm$^{{-1}}$ is what the study uses)")
ax.legend()
plt.show()

## Step 3 — PCA: is there structure at all?

791 features for 360 molecules is a wide, short matrix. Principal component analysis finds the
directions along which the spectra vary most, letting us look at the whole set in 2D.

PCA is **unsupervised** — it never sees the metal labels. So if the colors below separate at all,
that separation is genuinely in the spectra, not something we drew in.

In [ ]:
from sklearn.decomposition import PCA

is_metal = meta["metal"].values != "none"
pca = PCA(n_components=10).fit(X[is_metal])
coords = pca.transform(X[is_metal])
evr = pca.explained_variance_ratio_ * 100
sub = meta[is_metal]

fig, ax = plt.subplots(figsize=(7.5, 6))
for m in METAL_ORDER:
    s = sub["metal"].values == m
    ax.scatter(coords[s, 0], coords[s, 1], s=34, color=METAL_COLORS[m],
               label=METAL_LABELS[m], alpha=0.8, edgecolors="white", linewidths=0.5)
ax.set_xlabel(f"PC1 ({evr[0]:.1f}% of variance)")
ax.set_ylabel(f"PC2 ({evr[1]:.1f}% of variance)")
ax.set_title("Step 3 — PCA of the metal complexes")
ax.legend()
plt.show()

print(f"first 2 PCs explain {evr[:2].sum():.1f}% of the variance; first 10 explain {evr.sum():.1f}%")

Two things worth noticing, and both are honest negative results:

1. **PC1 and PC2 explain very little of the variance.** IR spectra vary in hundreds of weakly
   correlated ways; there is no dominant direction. A low number here is normal for spectroscopy
   and is *not* a sign something went wrong.
2. **The metals do not form clean clusters.** The dominant variation is peptide *length* and
   *sequence*, not the metal. Which is the real lesson: the metal signal is there, but it is a
   small perturbation on top of much larger variation — so we need a supervised model that is
   told what to look for.

We can ask *which wavenumbers* each component responds to by plotting its loading vector as if
it were itself a spectrum.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for k, (ax, color) in enumerate(zip(axes, ["#15807D", "#C1651B"])):
    ax.axhline(0, color="0.6", lw=1)
    ax.plot(WN, pca.components_[k], lw=1.4, color=color)
    ax.set_ylabel(f"PC{k + 1} loading")
    ax.set_title(f"PC{k + 1} — {evr[k]:.1f}% of variance", loc="left", fontweight="bold")
axes[0].set_xlim(WN.min(), WN.max())
axes[1].set_xlabel("wavenumber (cm$^{-1}$)")
annotate_bands(axes[0], fontsize=7)
plt.tight_layout()
plt.show()

## Step 4 — Train a classifier

A random forest on the 791 intensities. Two decisions worth understanding:

**Why a random forest and not a neural net?** With a few hundred training molecules, a forest is
the right capacity. It also handles correlated features gracefully (adjacent wavenumber bins are
extremely correlated) and needs almost no tuning to be reasonable.

**Why `class_weight="balanced_subsample"`?** So a class with fewer molecules is not quietly
ignored. Here the classes are balanced by construction, but the full study's are not.

We score with **macro-F1**: the F1 of each metal computed separately, then averaged unweighted.
That means getting Zn right counts exactly as much as getting Co right, even if one is rarer.
Chance is 0.25 for four balanced classes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, confusion_matrix

from irspectra.config import RF_PARAMS

short = is_metal & (meta["length"].values < 3)          # monomers + dimers
X_short, y_short = X[short], meta["metal"].values[short]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
rf = RandomForestClassifier(random_state=0, n_jobs=-1, **RF_PARAMS)
pred = cross_val_predict(rf, X_short, y_short, cv=cv)

macro = f1_score(y_short, pred, labels=METAL_ORDER, average="macro")
print(f"training molecules: {len(y_short)}")
print(f"out-of-fold macro-F1: {macro:.3f}   (chance = 0.25)")

### Where the mistakes are

A single number hides the interesting part. The confusion matrix below is **row-normalized**, so
each row sums to 1 and the diagonal is that metal's recall — "of all the real Cu²⁺ spectra, what
fraction did we call Cu²⁺?"

In [ ]:
cm = confusion_matrix(y_short, pred, labels=METAL_ORDER).astype(float)
cm /= cm.sum(axis=1, keepdims=True)                     # row-normalize -> recall on the diagonal

fig, ax = plt.subplots(figsize=(5.6, 5))
ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
labels = [METAL_LABELS[m] for m in METAL_ORDER]
ax.set_xticks(range(4), labels)
ax.set_yticks(range(4), labels)
for tick, m in zip(ax.get_xticklabels() + ax.get_yticklabels(), METAL_ORDER * 2):
    tick.set_color(METAL_COLORS[m]); tick.set_fontweight("bold")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm[i, j] > 0.5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Step 4 — per-metal confusion (out-of-fold)")
plt.show()

The off-diagonal structure is chemistry, not noise. **Co²⁺ and Ni²⁺ are the pair the model
confuses most.** They sit next to each other in the first transition series, have similar ionic
radii, and — critically — both commonly adopt octahedral coordination with these ligands. Their
metal–ligand stretching frequencies land almost on top of each other. Cu²⁺ is usually the
easiest: Jahn–Teller distortion gives its complexes a distinctive geometry, and hence a
distinctive vibrational signature.

### ⚠️ The mistake that would invalidate all of this

The full dataset has **many conformers per molecule** — the same peptide frozen in different
shapes. Their spectra are nearly identical.

If you split those rows at random, conformer #3 of a molecule lands in training and conformer #7
of the *same molecule* lands in test. The model doesn't have to learn chemistry; it can memorize
the molecule. You get a beautiful score that means nothing.

The fix is to split by **group**, where the group is the molecule:

```python
from sklearn.model_selection import StratifiedGroupKFold
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
for train_idx, test_idx in cv.split(X, y, groups=meta["molecule"]):
    ...     # every conformer of a molecule is on exactly one side of the split
```

Every modelling script in `pipeline/` does this. In *this* notebook the subset has one spectrum
per molecule, so grouping would be a no-op and we use plain `StratifiedKFold` — but the moment
you switch to `conformers.load_conformers()`, grouping stops being optional.

## Step 5 — The real test: does it transfer to longer peptides?

Everything so far trained and tested on monomers and dimers. The point of the project is
trimers — longer, more realistic, far more expensive to simulate.

So: train on short peptides only, then predict trimers the model has **never seen at any
length**. This is a zero-shot transfer test.

In [ ]:
long = is_metal & (meta["length"].values == 3)
X_long, y_long = X[long], meta["metal"].values[long]

scores = []
for seed in range(5):
    model = RandomForestClassifier(random_state=seed, n_jobs=-1, **RF_PARAMS).fit(X_short, y_short)
    scores.append(f1_score(y_long, model.predict(X_long), labels=METAL_ORDER, average="macro"))
zero_shot, zs_err = float(np.mean(scores)), float(np.std(scores))

print(f"in-domain (mono+di, out-of-fold): {macro:.3f}")
print(f"zero-shot on trimers:             {zero_shot:.3f} +/- {zs_err:.3f}")
print(f"length drop:                      {macro - zero_shot:.3f}")

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.bar(["in-domain\n(mono+di)", "zero-shot\n(trimers)"], [macro, zero_shot],
       yerr=[0, zs_err], color=["#2A78D6", "#EB6834"], capsize=6, width=0.55)
ax.axhline(0.25, ls=":", color="0.4", label="chance = 0.25")
ax.set_ylim(0, 1); ax.set_ylabel("macro-F1"); ax.legend()
ax.set_title("Step 5 — accuracy drops when the peptide grows")
plt.show()

**This gap is the project's central result.** The model keeps real skill — well above the 0.25
chance line, so metal identity does carry across lengths — but it loses a substantial chunk of
it.

Why? A trimer is not just a longer monomer. It has more backbone carbonyls competing for the
metal, more conformational freedom, and its amide bands are stronger and shifted. The model
learned what Ni²⁺ looks like *in the spectral context of a short peptide*, and that context
changed.

## Step 6 — How much trimer data would fix it?

Simulating trimers is expensive, so the practical question is: what is the *cheapest* amount of
trimer data that buys back most of the lost accuracy?

We hold out a fixed set of trimers as the test set, then grow the training set by adding trimer
molecules from a disjoint pool, and watch the curve.

In [ ]:
from sklearn.model_selection import train_test_split

GRID = [0, 20, 40, 60, 80, 100, 120]
rows = []
for seed in range(5):
    pool_idx, test_idx = train_test_split(np.arange(len(y_long)), test_size=0.25,
                                          random_state=seed, stratify=y_long)
    X_test, y_test = X_long[test_idx], y_long[test_idx]
    order = np.random.RandomState(seed).permutation(pool_idx)
    for n_added in GRID:
        add = order[:n_added]
        X_fit = np.vstack([X_short, X_long[add]]) if n_added else X_short
        y_fit = np.concatenate([y_short, y_long[add]]) if n_added else y_short
        model = RandomForestClassifier(random_state=seed, n_jobs=-1, **RF_PARAMS).fit(X_fit, y_fit)
        rows.append({"seed": seed, "n_added": n_added,
                     "macro_f1": f1_score(y_test, model.predict(X_test),
                                          labels=METAL_ORDER, average="macro")})

curve = pd.DataFrame(rows).groupby("n_added")["macro_f1"].agg(["mean", "std", "count"])
curve["ci95"] = 1.96 * curve["std"] / np.sqrt(curve["count"])
curve[["mean", "ci95"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(curve.index, curve["mean"], yerr=curve["ci95"], marker="o", lw=2,
            color="#1BAF7A", capsize=4, label="trained on mono+di + N trimers")
ax.axhline(curve["mean"].iloc[0], ls=":", color="#EB6834", label="zero-shot (N = 0)")
ax.set_xlabel("trimer molecules added to training")
ax.set_ylabel("macro-F1 on held-out trimers")
ax.set_title("Step 6 — how much long-peptide data do you actually need?")
ax.legend()
plt.show()

gain = curve["mean"].iloc[-1] - curve["mean"].iloc[0]
print(f"adding {GRID[-1]} trimers gains {gain:+.3f} macro-F1")

The shape matters more than the endpoint: the curve is **steep at first and then flattens**. Most
of the recoverable accuracy arrives with the first modest batch of trimers, and each additional
one buys less. That is what justifies the project's headline recommendation — anchor on cheap
short peptides, then top up with a small, diverse set of expensive long ones.

(With only ~120 trimers available here the curve is noisy. The full study runs this in 5%
increments over ~1,400 trimer molecules with 10 seeds, in `pipeline/step6_frontier.py`.)

---

## Recap

| Step | Idea | Takeaway |
|---|---|---|
| 1 | Look before you model | Length dominates the visual differences; metal is subtle |
| 2 | Featurization is a choice | FWHM 15 cm⁻¹ balances noise against band separation |
| 3 | Unsupervised structure | PCA does *not* separate metals — supervision is required |
| 4 | Train and inspect errors | ~0.76 macro-F1; Co/Ni is the hard pair, for real chemical reasons |
| 5 | Test the generalization you care about | Zero-shot to trimers drops to ~0.52 |
| 6 | Buy back the gap efficiently | Early trimers help most; returns diminish |

## Exercises

1. **Add the apo class.** We dropped `metal == "none"` throughout. Put it back as a fifth class
   (use `ALL_ORDER` / `ALL_COLORS`, which exist for exactly this). Is "is anything bound at all?"
   easier or harder than "which metal is bound?"
2. **Restrict the spectral range.** Refit Step 4 using only the metal–ligand region
   (`WN < 700`), then only the amide region (1300–1750). Which carries more of the metal signal?
   Does that match what the Step 3 loadings suggested?
3. **Break it on purpose.** In Step 6, sample the added trimers from the *test* set instead of
   the pool. Watch macro-F1 shoot up. This is the leakage failure mode from Step 4 — learn to
   recognise its signature.
4. **Change the featurization.** Re-broaden from `sticks` at FWHM 5 and 60 and rerun Step 4. How
   much does the score move? Is 15 cm⁻¹ actually a good choice?
5. **Go to full resolution.** Swap `processed.load_teaching()` for `conformers.load_conformers()`
   (no extra data needed — the conformer tables are committed too). You will now have many conformers per molecule,
   so you **must** switch to `StratifiedGroupKFold` with `groups=meta["molecule"]`. Compare the
   score with and without grouping — the difference is the leakage you would have reported.